# REGENIE GWAS PROCESSING CONGRADS

In [1]:
import pandas as pd
import numpy as np
import glob, os
import seaborn as sns
import matplotlib.pyplot as plt
from itertools import repeat

In [2]:
def read_sumstats_REGENIE_CON(fn, no_phenos, drop_cols=None):
    """
    Reads sumstats
    
    Converts regenie output to MRP format (PLINK-like)
    - add necessary column
    - change column names
    
    """
    #load data
    data = pd.read_csv(fn, sep=" ")
    
    #get non-logP for FUMA
    if no_phenos > 1:
        for i in range(1, no_phenos+1, 1):
            data["P.Y{}".format(i)] = np.power(10, -data["LOG10P.Y{}".format(i)])
    else: 
        data["P"] = np.power(10, -data["LOG10P"])
        
    if drop_cols:
        data.drop(drop_cols, axis=1, inplace=True)
    
    #use unique ID
    return data.set_index("ID")

def get_cumulative_pos(df):
    running_pos = float(0)

    cumulative_pos = []

    for chrom, group_df in df.groupby('CHROM'):
        chr_df = np.add(group_df['GENPOS'].astype("int64"), running_pos)
        cumulative_pos.append(chr_df)
        running_pos += float(group_df['GENPOS'].max())

    df['cumulative_pos'] = pd.concat(cumulative_pos)
    
    return df

def data_loader_REGENIE_all_chromosomes_CON(regenie_path, chr_nos, no_phenos):
    """
    """
    regenie_paths = sorted(glob.glob(regenie_path))
    
    no_phenos = [no_phenos] * len(regenie_paths)
    
    print(regenie_paths)
    
    df = pd.concat(map(read_sumstats_REGENIE_CON, regenie_paths, no_phenos))
    
    df = get_cumulative_pos(df)
    
    return df

def read_sumstats_REGENIE(fn, drop_cols = None):
    """
    Reads sumstats
    
    Converts regenie output to MRP format (PLINK-like)
    - add necessary column
    - change column names
    
    """
    #load data
    data = pd.read_csv(fn, sep=" ")
    
    #get non-logP for FUMA
    data["P"] = np.power(10, -data["LOG10P"])
    
    if drop_cols:
        data.drop(drop_cols, axis=1, inplace=True)
    
    #use unique ID
    return data.set_index("ID")


def data_loader_REGENIE_all_chromosomes(regenie_path, chr_nos):
    """
    """
    regenie_paths = sorted(glob.glob(regenie_path))
    
    print(regenie_paths)
    
    df = pd.concat(map(read_sumstats_REGENIE, regenie_paths))
    
    df = get_cumulative_pos(df)
    
    return df



def manhattan(df, ax, figargs={"figsize":(14,8),"dpi":300}):
    
    g = sns.scatterplot(
        data = df,
        x = 'cumulative_pos',
        y = 'LOG10P',
        hue = 'CHR',
        palette = ['indianred', 'mediumblue'] * 11 + ['indianred'],
        linewidth=0,
        s=6,
        zorder=2,
        ax=ax,
        legend=None,
        style=None
    )
    
    alpha = -np.log10(0.05/10000000)
    
    ax.plot([np.min(df["cumulative_pos"]), np.max(df["cumulative_pos"])], [alpha, alpha], "--", color="navy")

    ax.set_xlabel('Chromosome')
    ax.set_ylabel('-Log10 P value')

    ax.set_xticks(df.groupby('CHROM')['cumulative_pos'].median())
    ax.tick_params(which='major', width=0.3, length=2, labelsize=8)

    ax.set_xticklabels(df['CHROM'].unique())
    
    ax.set_ylim(0, 50)
    
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_visible(True)
    ax.spines["bottom"].set_visible(True)


In [3]:
def miami(df1, df2, y_dsc1, y_dsc2, title_str, alpha, figargs= {"figsize":(14,10),"dpi":150}):
    """
    Big plotting function to make a gene-based Miami plot
    """
    
    df1 = df1.loc[df1["LOG10P"] > 1, :]
    df2 = df2.loc[df2["LOG10P"] > 1, :]
    
    no_genes = len(df1)
    
    df1 = get_cumulative_pos(df1)
    df2 = get_cumulative_pos(df2)
    
    max_p = np.max([np.max(df1["LOG10P"]), np.max(df2["LOG10P"])])
    
    fig, (ax1, ax2) = plt.subplots(2, 1, 
        gridspec_kw={'height_ratios': [1, 1]},**figargs)
    plt.subplots_adjust(hspace=0.16)
    
    g = sns.scatterplot(data=df1,
                        #x=df1.index.values,
                        x="cumulative_pos",
                        y="LOG10P",
                        hue = 'CHROM',
                        palette = ['#00786A', '#A4D3EE'] * 12,
                        legend=None,
                        style=None,
                        s=6,
                        linewidth=0,
                        zorder=2,
                        ax=ax1,
                        edgecolor="black")     

    g = sns.scatterplot(data=df2,
                        #x=df2.index.values,
                        x="cumulative_pos",
                        y="LOG10P",
                        hue = 'CHROM',
                        palette = ['#00786A', '#A4D3EE'] * 12,
                        legend=None,
                        style=None,
                        s=6,
                        linewidth=0,
                        zorder=2,
                        ax=ax2,
                        edgecolor="black") 
    
    ax1.set_xticks(df1.groupby('CHROM')['cumulative_pos'].median())
    ax1.tick_params(which='major', width=0.3, length=2, labelsize=8)
    ax1.set_xticklabels(sorted(df1['CHROM'].unique()))
    ax1.set_xlabel('Chromosome')
    ax1.set_ylabel(y_dsc1)
    ax1.set_ylim([0, max_p])

    if alpha is not None:
        ax1.plot([np.min(df1["cumulative_pos"]), np.max(df1["cumulative_pos"])], [alpha, alpha], "--", color="dimgrey")
    
    ax2.set_xticks(df2.groupby('CHROM')['cumulative_pos'].median())
    ax2.set_ylim([0, max_p])
    ax2.set_xlim(ax1.get_xlim())
    ax2.set_xlabel('')
    ax2.set_xticklabels([])
    
    ax2.xaxis.set_ticks_position("top")
    ax2.set_ylabel(y_dsc2)
    
    if alpha is not None:
        ax2.plot([np.min(df2["cumulative_pos"]), np.max(df2["cumulative_pos"])], [alpha, alpha], "--", color="dimgrey")
   
    ax2.invert_yaxis()
    
    ax1.spines["top"].set_visible(False)
    ax1.spines["right"].set_visible(False)
    ax1.spines["left"].set_visible(True)
    ax1.spines["bottom"].set_visible(True)
    
    ax2.spines["top"].set_visible(True)
    ax2.spines["right"].set_visible(False)
    ax2.spines["left"].set_visible(True)
    ax2.spines["bottom"].set_visible(False)
    
    #if alpha is not None:
    #    for gene in df1[df1[y_name1] > alpha].index.values:
    #        ax1.text(df1.loc[gene, "cumulative_pos"]+1, df1.loc[gene, y_name1]+0.25, df1.loc[gene, "gene_name"], horizontalalignment='left', size='small', color='black') #, weight='semibold')
    #    for gene in df2[df2[y_name2] > alpha].index.values:
    #        ax2.text(df2.loc[gene, "cumulative_pos"]+1, df2.loc[gene, y_name2]+0.25, df2.loc[gene, "gene_name"], horizontalalignment='left', size='small', color='black') #, weight='semibold')
    
    fig.suptitle(title_str)

In [3]:
results_path = "/data/clusterfs/lag/users/jitame/CONGRADS/geno/regenie/st2_out_gwas/"
workspace_path = "/data/workspaces/lag/workspaces/lg-ukbiobank/projects/CONGRADS_rest/"

#oneliner to get all results files
phenos = [os.path.split(fn)[1].replace("c1", "c*") for fn in sorted(glob.glob(os.path.join(results_path, "c1/CONGRADS_gwas_cmap*_65k_*.regenie")))]

chromosomes = list(range(1, 23, 1)) + ["X", "XY"]

sumstats_dict = dict()

print(phenos)

for pheno in phenos:
    print(pheno)
    
    fn = os.path.join(results_path, "c*", pheno)
    sumstats_pheno = data_loader_REGENIE_all_chromosomes(fn,
                                                     chromosomes)
    sumstats_dict.update({pheno: sumstats_pheno})
    out_fn = os.path.join(workspace_path, pheno.replace("_c*", ""))
    sumstats_pheno.to_csv(out_fn, sep="\t")


['CONGRADS_gwas_cmap_65k_multi_c*.regenie', 'CONGRADS_gwas_cmaps_65k_c*_melodic_g1_cmap_lifg_oic_0.regenie', 'CONGRADS_gwas_cmaps_65k_c*_melodic_g1_cmap_lifg_oic_1.regenie', 'CONGRADS_gwas_cmaps_65k_c*_melodic_g1_cmap_lifg_oic_2.regenie', 'CONGRADS_gwas_cmaps_65k_c*_melodic_g1_cmap_lstg_oic_0.regenie', 'CONGRADS_gwas_cmaps_65k_c*_melodic_g1_cmap_lstg_oic_1.regenie', 'CONGRADS_gwas_cmaps_65k_c*_melodic_g1_cmap_lstg_oic_2.regenie', 'CONGRADS_gwas_cmaps_65k_c*_melodic_g2_cmap_lifg_oic_0.regenie', 'CONGRADS_gwas_cmaps_65k_c*_melodic_g2_cmap_lifg_oic_1.regenie', 'CONGRADS_gwas_cmaps_65k_c*_melodic_g2_cmap_lstg_oic_0.regenie', 'CONGRADS_gwas_cmaps_65k_c*_melodic_g2_cmap_lstg_oic_1.regenie', 'CONGRADS_gwas_cmaps_65k_c*_melodic_g2_cmap_lstg_oic_2.regenie']
CONGRADS_gwas_cmap_65k_multi_c*.regenie
['/data/clusterfs/lag/users/jitame/CONGRADS/geno/regenie/st2_out_gwas/c1/CONGRADS_gwas_cmap_65k_multi_c1.regenie', '/data/clusterfs/lag/users/jitame/CONGRADS/geno/regenie/st2_out_gwas/c10/CONGRADS_gwas

CONGRADS_gwas_cmaps_65k_c*_melodic_g1_cmap_lifg_oic_2.regenie
['/data/clusterfs/lag/users/jitame/CONGRADS/geno/regenie/st2_out_gwas/c1/CONGRADS_gwas_cmaps_65k_c1_melodic_g1_cmap_lifg_oic_2.regenie', '/data/clusterfs/lag/users/jitame/CONGRADS/geno/regenie/st2_out_gwas/c10/CONGRADS_gwas_cmaps_65k_c10_melodic_g1_cmap_lifg_oic_2.regenie', '/data/clusterfs/lag/users/jitame/CONGRADS/geno/regenie/st2_out_gwas/c11/CONGRADS_gwas_cmaps_65k_c11_melodic_g1_cmap_lifg_oic_2.regenie', '/data/clusterfs/lag/users/jitame/CONGRADS/geno/regenie/st2_out_gwas/c12/CONGRADS_gwas_cmaps_65k_c12_melodic_g1_cmap_lifg_oic_2.regenie', '/data/clusterfs/lag/users/jitame/CONGRADS/geno/regenie/st2_out_gwas/c13/CONGRADS_gwas_cmaps_65k_c13_melodic_g1_cmap_lifg_oic_2.regenie', '/data/clusterfs/lag/users/jitame/CONGRADS/geno/regenie/st2_out_gwas/c14/CONGRADS_gwas_cmaps_65k_c14_melodic_g1_cmap_lifg_oic_2.regenie', '/data/clusterfs/lag/users/jitame/CONGRADS/geno/regenie/st2_out_gwas/c15/CONGRADS_gwas_cmaps_65k_c15_melodic_g1

CONGRADS_gwas_cmaps_65k_c*_melodic_g1_cmap_lstg_oic_2.regenie
['/data/clusterfs/lag/users/jitame/CONGRADS/geno/regenie/st2_out_gwas/c1/CONGRADS_gwas_cmaps_65k_c1_melodic_g1_cmap_lstg_oic_2.regenie', '/data/clusterfs/lag/users/jitame/CONGRADS/geno/regenie/st2_out_gwas/c10/CONGRADS_gwas_cmaps_65k_c10_melodic_g1_cmap_lstg_oic_2.regenie', '/data/clusterfs/lag/users/jitame/CONGRADS/geno/regenie/st2_out_gwas/c11/CONGRADS_gwas_cmaps_65k_c11_melodic_g1_cmap_lstg_oic_2.regenie', '/data/clusterfs/lag/users/jitame/CONGRADS/geno/regenie/st2_out_gwas/c12/CONGRADS_gwas_cmaps_65k_c12_melodic_g1_cmap_lstg_oic_2.regenie', '/data/clusterfs/lag/users/jitame/CONGRADS/geno/regenie/st2_out_gwas/c13/CONGRADS_gwas_cmaps_65k_c13_melodic_g1_cmap_lstg_oic_2.regenie', '/data/clusterfs/lag/users/jitame/CONGRADS/geno/regenie/st2_out_gwas/c14/CONGRADS_gwas_cmaps_65k_c14_melodic_g1_cmap_lstg_oic_2.regenie', '/data/clusterfs/lag/users/jitame/CONGRADS/geno/regenie/st2_out_gwas/c15/CONGRADS_gwas_cmaps_65k_c15_melodic_g1

CONGRADS_gwas_cmaps_65k_c*_melodic_g2_cmap_lstg_oic_0.regenie
['/data/clusterfs/lag/users/jitame/CONGRADS/geno/regenie/st2_out_gwas/c1/CONGRADS_gwas_cmaps_65k_c1_melodic_g2_cmap_lstg_oic_0.regenie', '/data/clusterfs/lag/users/jitame/CONGRADS/geno/regenie/st2_out_gwas/c10/CONGRADS_gwas_cmaps_65k_c10_melodic_g2_cmap_lstg_oic_0.regenie', '/data/clusterfs/lag/users/jitame/CONGRADS/geno/regenie/st2_out_gwas/c11/CONGRADS_gwas_cmaps_65k_c11_melodic_g2_cmap_lstg_oic_0.regenie', '/data/clusterfs/lag/users/jitame/CONGRADS/geno/regenie/st2_out_gwas/c12/CONGRADS_gwas_cmaps_65k_c12_melodic_g2_cmap_lstg_oic_0.regenie', '/data/clusterfs/lag/users/jitame/CONGRADS/geno/regenie/st2_out_gwas/c13/CONGRADS_gwas_cmaps_65k_c13_melodic_g2_cmap_lstg_oic_0.regenie', '/data/clusterfs/lag/users/jitame/CONGRADS/geno/regenie/st2_out_gwas/c14/CONGRADS_gwas_cmaps_65k_c14_melodic_g2_cmap_lstg_oic_0.regenie', '/data/clusterfs/lag/users/jitame/CONGRADS/geno/regenie/st2_out_gwas/c15/CONGRADS_gwas_cmaps_65k_c15_melodic_g2

In [3]:
results_path = "/data/clusterfs/lag/users/jitame/CONGRADS/geno/regenie/st2_out_gwas/"
workspace_path = "/data/workspaces/lag/workspaces/lg-ukbiobank/projects/CONGRADS_rest/"

#oneliner to get all results files
phenos = [os.path.split(fn)[1].replace("c1", "c*") for fn in sorted(glob.glob(os.path.join(results_path, "c1/CONGRADS_gwas_cmap*_65k_*.regenie"")))]

chromosomes = list(range(1, 23, 1)) + ["X", "XY"]

sumstats_dict = dict()

print(phenos)

for pheno in phenos:
    print(pheno)
    
    fn = os.path.join(results_path, "c*", pheno)
    sumstats_pheno = data_loader_REGENIE_all_chromosomes(fn,
                                                     chromosomes)
    sumstats_dict.update({pheno: sumstats_pheno})
    out_fn = os.path.join(workspace_path, pheno.replace("_c*", ""))
    sumstats_pheno.to_csv(out_fn, sep="\t")


['CONGRADS_gwas_g12_pmaps_65k_multi_c*.regenie', 'CONGRADS_gwas_g1_pmaps_65k_multi_c*.regenie', 'CONGRADS_gwas_g2_pmaps_65k_multi_c*.regenie']
CONGRADS_gwas_g12_pmaps_65k_multi_c*.regenie
['/data/clusterfs/lag/users/jitame/CONGRADS/geno/regenie/st2_out_gwas/c1/CONGRADS_gwas_g12_pmaps_65k_multi_c1.regenie', '/data/clusterfs/lag/users/jitame/CONGRADS/geno/regenie/st2_out_gwas/c10/CONGRADS_gwas_g12_pmaps_65k_multi_c10.regenie', '/data/clusterfs/lag/users/jitame/CONGRADS/geno/regenie/st2_out_gwas/c11/CONGRADS_gwas_g12_pmaps_65k_multi_c11.regenie', '/data/clusterfs/lag/users/jitame/CONGRADS/geno/regenie/st2_out_gwas/c12/CONGRADS_gwas_g12_pmaps_65k_multi_c12.regenie', '/data/clusterfs/lag/users/jitame/CONGRADS/geno/regenie/st2_out_gwas/c13/CONGRADS_gwas_g12_pmaps_65k_multi_c13.regenie', '/data/clusterfs/lag/users/jitame/CONGRADS/geno/regenie/st2_out_gwas/c14/CONGRADS_gwas_g12_pmaps_65k_multi_c14.regenie', '/data/clusterfs/lag/users/jitame/CONGRADS/geno/regenie/st2_out_gwas/c15/CONGRADS_gwas_

ParserError: Error tokenizing data. C error: Calling read(nbytes) on source failed. Try engine='python'.

In [9]:
sig = -np.log10(0.05/1000000)
print("Number of hits:")
print(np.sum((sumstats_dict[phenos[0]]["LOG10P"] > sig)))
print("Biggest hits:")
print(sumstats_dict[phenos[0]].sort_values(by="LOG10P", inplace=False, ascending=False).head(n=50))
print(sumstats_dict[phenos[0]].sort_values(by="LOG10P", inplace=False, ascending=False).head(n=155).tail(n=50))

Number of hits:
155
Biggest hits:
            CHROM     GENPOS ALLELE0 ALLELE1      MAC    A1FREQ      N  \
ID                                                                       
rs3891783      10   96015793       C       G  35884.9  0.433005  41437   
rs57866767     10   96023077       T       C  35887.2  0.433033  41437   
rs10786156     10   96014622       C       G  35887.2  0.433033  41437   
rs11187838     10   96038686       G       A  35853.7  0.432629  41437   
rs2274224      10   96039597       G       C  35872.0  0.432850  41437   
rs71481917     10  134297801       C       T  36250.8  0.437420  41437   
rs34102287     10  134331173       C       T  35676.4  0.430490  41437   
rs7080472      10   96012950       G       T  35070.1  0.423173  41437   
rs7915907      10  134316231       G       A  36521.1  0.440683  41437   
rs12762160     10  134330662       C       T  36485.0  0.440247  41437   
rs57472801     10  134322756       A       G  36493.3  0.440347  41437   
rs11